# Hidden Markov Models: When Clusters Have Memory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/unsupervised/hidden_markov_models.ipynb)

This notebook implements Hidden Markov Models from first principles and applies them to stock market regime detection.

**What you'll learn:**
- Markov chains and transition matrices
- The HMM structure: hidden states, emissions, and parameters (A, B, π)
- The Forward algorithm for computing sequence likelihoods
- The Viterbi algorithm for finding the most likely state sequence
- Applying Gaussian HMMs to detect market regimes
- How HMMs compare to K-Means on the same data

In [ ]:
# Install dependencies (uncomment in Colab)
# !pip install hmmlearn yfinance -q

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

## 1. Markov Chains: Adding Memory to Randomness

A Markov chain is a sequence of random variables where the next state depends **only** on the current state (the Markov property). We define it with a **transition matrix** A.

In [ ]:
# A simple weather Markov chain: Sunny (0), Cloudy (1), Rainy (2)
states = ['Sunny', 'Cloudy', 'Rainy']
A = np.array([
    [0.7, 0.2, 0.1],  # From Sunny
    [0.3, 0.4, 0.3],  # From Cloudy
    [0.2, 0.3, 0.5],  # From Rainy
])
pi = np.array([0.5, 0.3, 0.2])  # Initial state probabilities

def simulate_markov_chain(A, pi, n_steps, seed=42):
    """Simulate a Markov chain."""
    rng = np.random.default_rng(seed)
    state_seq = [rng.choice(len(pi), p=pi)]
    for _ in range(n_steps - 1):
        state_seq.append(rng.choice(len(pi), p=A[state_seq[-1]]))
    return np.array(state_seq)

chain = simulate_markov_chain(A, pi, n_steps=50)

colours_weather = ['#f39c12', '#95a5a6', '#3498db']
fig, ax = plt.subplots(figsize=(12, 2.5))
for t, s in enumerate(chain):
    ax.bar(t, 1, color=colours_weather[s], width=1.0, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Time step')
ax.set_yticks([])
ax.set_title('Simulated Weather Markov Chain')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=s) for c, s in zip(colours_weather, states)],
          loc='upper right', ncol=3)
plt.tight_layout()
plt.show()

# Check: state frequencies should approximate the stationary distribution
long_chain = simulate_markov_chain(A, pi, n_steps=10000)
freqs = np.bincount(long_chain, minlength=3) / len(long_chain)
print(f"Empirical frequencies: {dict(zip(states, freqs.round(3)))}")

Notice the **clustering** — rainy days tend to follow rainy days, sunny days cluster together. This temporal dependency is exactly what K-Means and GMMs **cannot** capture.

## 2. Hidden Markov Models: When You Can't See the States

In an HMM, the Markov chain runs in the **hidden** layer. You only observe **emissions** that depend on the hidden state.

**The Umbrella Example:** You can't see the weather (hidden), but you can see whether your colleague carries an umbrella (observed).

An HMM has three sets of parameters λ = (A, B, π):
- **A** — Transition matrix: P(next state | current state)
- **B** — Emission probabilities: P(observation | state)
- **π** — Initial state distribution

In [ ]:
# HMM parameters for the umbrella problem
# Hidden states: Sunny (0), Rainy (1)
# Observations: No umbrella (0), Umbrella (1)

A_hmm = np.array([
    [0.7, 0.3],  # Sunny → Sunny, Sunny → Rainy
    [0.4, 0.6],  # Rainy → Sunny, Rainy → Rainy
])

B_hmm = np.array([
    [0.9, 0.1],  # Sunny → P(no umbrella), P(umbrella)
    [0.2, 0.8],  # Rainy → P(no umbrella), P(umbrella)
])

pi_hmm = np.array([0.6, 0.4])  # P(Sunny), P(Rainy) at t=0

# An observed sequence: did the colleague carry an umbrella?
observations = np.array([1, 1, 0, 1, 0, 0, 1, 1, 1, 0])  # 1=umbrella, 0=no umbrella
obs_labels = ['Umbrella' if o else 'No umbrella' for o in observations]
print(f"Observed: {obs_labels}")
print(f"Question: What was the weather (hidden state) each day?")

## 3. The Forward Algorithm: How Likely Is This Sequence?

**Task 1 (Evaluation):** Given model parameters λ and an observation sequence O, compute P(O | λ).

The forward variable α_t(i) = P(O_1, ..., O_t, Q_t = i | λ) — the probability of seeing the first t observations AND being in state i at time t.

In [ ]:
def forward_algorithm(observations, A, B, pi):
    """
    Forward algorithm for HMMs.

    Args:
        observations: (T,) array of observation indices
        A: (K, K) transition matrix
        B: (K, M) emission matrix
        pi: (K,) initial state distribution

    Returns:
        alpha: (T, K) forward variables
        log_likelihood: log P(O | λ)
    """
    T = len(observations)
    K = len(pi)
    alpha = np.zeros((T, K))

    # Initialisation: α_1(i) = π_i * B_i(O_1)
    alpha[0] = pi * B[:, observations[0]]

    # Recursion: α_t(j) = [Σ_i α_{t-1}(i) * A_ij] * B_j(O_t)
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * B[:, observations[t]]

    # Termination: P(O|λ) = Σ_i α_T(i)
    log_likelihood = np.log(alpha[-1].sum())

    return alpha, log_likelihood

alpha, log_lik = forward_algorithm(observations, A_hmm, B_hmm, pi_hmm)
print(f"Log-likelihood of the observation sequence: {log_lik:.4f}")
print(f"P(O|λ) = {np.exp(log_lik):.6e}")

# Forward variables give us P(state | observations up to time t)
state_probs = alpha / alpha.sum(axis=1, keepdims=True)
print(f"\nFiltered state probabilities (P(state | O_1:t)):")
print(f"{'Day':>4s}  {'Observed':>12s}  {'P(Sunny)':>10s}  {'P(Rainy)':>10s}")
for t in range(len(observations)):
    print(f"{t+1:4d}  {obs_labels[t]:>12s}  {state_probs[t,0]:10.4f}  {state_probs[t,1]:10.4f}")

## 4. The Viterbi Algorithm: Most Likely State Sequence

**Task 2 (Decoding):** Find the single most likely hidden state sequence given the observations.

In [ ]:
def viterbi(observations, A, B, pi):
    """
    Viterbi algorithm — find the most likely state sequence.

    Args:
        observations: (T,) array of observation indices
        A: (K, K) transition matrix
        B: (K, M) emission matrix
        pi: (K,) initial state distribution

    Returns:
        best_path: (T,) most likely state sequence
        log_prob: log probability of the best path
    """
    T = len(observations)
    K = len(pi)

    # Work in log space to avoid underflow
    log_A = np.log(A)
    log_B = np.log(B)
    log_pi = np.log(pi)

    # δ_t(j) = max log probability of any path ending in state j at time t
    delta = np.zeros((T, K))
    psi = np.zeros((T, K), dtype=int)  # backpointers

    # Initialisation
    delta[0] = log_pi + log_B[:, observations[0]]

    # Recursion
    for t in range(1, T):
        for j in range(K):
            scores = delta[t-1] + log_A[:, j]
            psi[t, j] = np.argmax(scores)
            delta[t, j] = scores[psi[t, j]] + log_B[j, observations[t]]

    # Backtrack
    best_path = np.zeros(T, dtype=int)
    best_path[-1] = np.argmax(delta[-1])
    log_prob = delta[-1, best_path[-1]]

    for t in range(T-2, -1, -1):
        best_path[t] = psi[t+1, best_path[t+1]]

    return best_path, log_prob

best_path, log_prob = viterbi(observations, A_hmm, B_hmm, pi_hmm)

hidden_states_labels = ['Sunny', 'Rainy']
print(f"Viterbi path (most likely weather sequence):")
print(f"{'Day':>4s}  {'Observed':>12s}  {'Hidden State':>12s}")
for t in range(len(observations)):
    print(f"{t+1:4d}  {obs_labels[t]:>12s}  {hidden_states_labels[best_path[t]]:>12s}")

In [ ]:
# Visualise: Forward probabilities vs Viterbi path
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

days = range(1, len(observations) + 1)

# Forward probabilities
ax = axes[0]
ax.bar(days, state_probs[:, 0], color='#f39c12', alpha=0.8, label='P(Sunny)')
ax.bar(days, state_probs[:, 1], bottom=state_probs[:, 0], color='#3498db', alpha=0.8, label='P(Rainy)')
ax.set_ylabel('Probability')
ax.set_title('Forward Algorithm: Filtered State Probabilities')
ax.legend(loc='upper right')

# Viterbi path
ax = axes[1]
viterbi_colours = ['#f39c12' if s == 0 else '#3498db' for s in best_path]
ax.bar(days, [1]*len(days), color=viterbi_colours, edgecolor='white', linewidth=0.5)
for t, (o, s) in enumerate(zip(obs_labels, best_path)):
    ax.text(t+1, 0.5, o.split()[0][:3], ha='center', va='center', fontsize=8, fontweight='bold')
ax.set_ylabel('State')
ax.set_xlabel('Day')
ax.set_title('Viterbi: Most Likely State Sequence')
ax.set_yticks([])

plt.tight_layout()
plt.show()

## 5. Stock Market Regime Detection

Now let's apply HMMs to a real problem: detecting **market regimes** (bull, bear, sideways) in stock price data.

We'll use a Gaussian HMM from `hmmlearn`, where emissions are continuous (daily returns and volume) instead of discrete.

In [ ]:
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

# Download S&P 500 data
ticker = yf.Ticker('^GSPC')
df = ticker.history(start='2018-01-01', end='2024-01-01')

# Features: daily return and scaled volume
df['Return'] = df['Close'].pct_change()
df = df[df['Volume'] > 0]  # drop zero-volume days (holidays etc.)
df['Log_Volume'] = np.log(df['Volume'])
df = df.dropna()

# Remove any remaining inf/nan
df = df[np.isfinite(df['Return']) & np.isfinite(df['Log_Volume'])]

features = df[['Return', 'Log_Volume']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print(f"Data: S&P 500, {len(df)} trading days ({df.index[0].date()} to {df.index[-1].date()})")
print(f"Features: daily return, log volume (standardised)")

In [ ]:
# Fit a 3-state Gaussian HMM
model = GaussianHMM(n_components=3, covariance_type='full', n_iter=200, random_state=42)
model.fit(X_scaled)

hidden_states = model.predict(X_scaled)
posteriors = model.predict_proba(X_scaled)

# Print regime characteristics
print("Transition Matrix:")
print(np.round(model.transmat_, 3))
print()

# Map regimes to interpretable labels based on mean return
regime_returns = [features[hidden_states == k, 0].mean() for k in range(3)]
order = np.argsort(regime_returns)  # low return → high return
regime_names = {order[0]: 'Bear', order[1]: 'Sideways', order[2]: 'Bull'}

for k in range(3):
    mask = hidden_states == k
    mean_ret = features[mask, 0].mean() * 252  # annualised
    vol = features[mask, 0].std() * np.sqrt(252)  # annualised
    pct = mask.sum() / len(mask) * 100
    print(f"Regime {k} ({regime_names[k]:>8s}): ann. return = {mean_ret:+.1%}, vol = {vol:.1%}, {pct:.0f}% of days")

In [ ]:
# Visualise: S&P 500 price coloured by regime
regime_colours = {order[0]: '#e74c3c', order[1]: '#f39c12', order[2]: '#2ecc71'}

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

# Price chart with regime colours
ax = axes[0]
dates = df.index
close = df['Close'].values
for k in range(3):
    mask = hidden_states == k
    ax.scatter(dates[mask], close[mask], c=regime_colours[k], s=3, alpha=0.7,
              label=f'{regime_names[k]}')
ax.set_ylabel('S&P 500 Close')
ax.set_title('S&P 500 Market Regimes Detected by HMM')
ax.legend(loc='upper left', markerscale=5)
ax.grid(True, alpha=0.3)

# Posterior probabilities
ax = axes[1]
bottom = np.zeros(len(dates))
for k in [order[2], order[1], order[0]]:  # stack bull first
    ax.fill_between(dates, bottom, bottom + posteriors[:, k],
                    color=regime_colours[k], alpha=0.8, label=regime_names[k])
    bottom += posteriors[:, k]
ax.set_ylabel('P(regime)')
ax.set_xlabel('Date')
ax.set_title('Posterior Regime Probabilities')
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 6. K-Means vs HMM on the Same Data

Let's compare what happens when we cluster the same features with K-Means (which ignores time) vs HMM (which models temporal dependency).

In [ ]:
from sklearn.cluster import KMeans

# K-Means on the same features
km = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = km.fit_predict(X_scaled)

# Map K-Means labels by mean return for comparison
km_returns = [features[km_labels == k, 0].mean() for k in range(3)]
km_order = np.argsort(km_returns)
km_names = {km_order[0]: 'Bear', km_order[1]: 'Sideways', km_order[2]: 'Bull'}
km_colours = {km_order[0]: '#e74c3c', km_order[1]: '#f39c12', km_order[2]: '#2ecc71'}

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# K-Means
ax = axes[0]
for k in range(3):
    mask = km_labels == k
    ax.scatter(dates[mask], close[mask], c=km_colours[k], s=3, alpha=0.7, label=km_names[k])
ax.set_ylabel('S&P 500 Close')
ax.set_title('K-Means: Clusters Based on Features Only (ignores time)')
ax.legend(loc='upper left', markerscale=5)
ax.grid(True, alpha=0.3)

# HMM
ax = axes[1]
for k in range(3):
    mask = hidden_states == k
    ax.scatter(dates[mask], close[mask], c=regime_colours[k], s=3, alpha=0.7, label=regime_names[k])
ax.set_ylabel('S&P 500 Close')
ax.set_title('HMM: Regimes with Temporal Dependency')
ax.legend(loc='upper left', markerscale=5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Count regime switches
km_switches = np.sum(np.diff(km_labels) != 0)
hmm_switches = np.sum(np.diff(hidden_states) != 0)
print(f"K-Means regime switches: {km_switches}")
print(f"HMM regime switches:     {hmm_switches}")
print(f"HMM produces {km_switches/hmm_switches:.1f}x fewer switches — smoother, more persistent regimes")

## 7. The Unified View: K-Means → GMM → HMM

| | K-Means | GMM | HMM |
|---|---------|-----|-----|
| **Responsibilities** | Hard (0 or 1) | Soft (probabilities) | Soft (probabilities) |
| **Data assumption** | i.i.d. | i.i.d. | Sequentially dependent |
| **Cluster shape** | Spherical (isotropic) | Elliptical (full covariance) | Elliptical (full covariance) |
| **Objective** | Distortion | Log-likelihood | Log-likelihood |
| **Fitting** | Lloyd's (EM, hard) | EM (soft) | Baum-Welch (EM for sequences) |

K-Means and GMMs assume observations are **independent**. HMMs capture the fact that market regimes **persist** — a bull market today makes a bull market tomorrow more likely.

## Exercises

1. **Number of states** — Fit HMMs with 2, 3, 4, and 5 states. Use BIC (`-2 * model.score(X) * len(X) + n_params * np.log(len(X))`) to choose the best K.

2. **Different stocks** — Apply the same HMM to a volatile stock (e.g., Tesla) vs a stable one (e.g., Johnson & Johnson). How do the regimes differ?

3. **Implement Baum-Welch** — Extend the forward algorithm with a backward pass and implement the parameter update equations from scratch.

4. **Regime-conditional trading** — Compute the Sharpe ratio within each detected regime. Could you build a trading strategy that reduces exposure during bear regimes?

5. **Discrete HMM** — Discretise the returns into bins (e.g., large drop, small drop, flat, small gain, large gain) and fit a discrete HMM. Compare with the Gaussian version.